<a href="https://colab.research.google.com/github/Sunitha-k2006/sunitha-codeboosters-internship-2026/blob/main/Phase_01_Data_Engineering/Day_04_BigData_PySpark_Architecture/Copy_of_Day_04_Machine_Learning_and_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
a=pd.read_csv("/content/SOCR-HeightWeight.csv")


In [ ]:
a.head()

,Index,Height(Inches),Weight(Pounds)
0,1,65.78331,112.9925
1,2,71.51521,136.4873
2,3,69.39874,153.0269
3,4,68.21660,142.3354
4,5,67.78781,144.2971


In [ ]:
x=a[['Height(Inches)']]
y=a[['Weight(Pounds)']]

In [ ]:
a.isnull().any()
a.isnull().sum()

,0
Index,0
Height(Inches),0
Weight(Pounds),0


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)
#splits the data for training and testing - 4 components
#test_size -> the percentage of data to be used for testing

In [ ]:
x_train

,Height(Inches)
9259,70.55747
1442,70.04554
5654,71.27290
9187,67.47051
18572,65.94714
...,...
20450,68.19770
13072,67.29233
24060,71.95831
23981,66.12776


In [ ]:
#select and import model----
from sklearn.linear_model import LinearRegression
model=LinearRegression()

#to train the model
model.fit(x_train,y_train)



LinearRegression()

In [ ]:
#to test the data - the output in the model predicted value
model.predict(x_test)

array([[135.73631407],
       [124.15722521],
       [133.15417259],
       ...,
       [114.03541602],
       [125.57086654],
       [126.80949092]])

In [ ]:
y_pred=model.predict(x_test)

In [ ]:
# we use this to minus the prdeicted answer with y test
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score


In [ ]:
mean_absolute_error(y_test,y_pred)

8.05958066569989

In [ ]:
mean_squared_error(y_test,y_pred)

101.93216170118505

In [ ]:
r2_score(y_test,y_pred)

0.2459172194533752

In [ ]:
import joblib

In [ ]:
joblib.dump(model,"linear.pkl")

['linear.pkl']

In [ ]:
!pip install pyspark --quiet
print("PySpark installation complete!")

PySpark installation complete!


In [ ]:
from pyspark.sql import SparkSession

from pyspark.sql import functions as F

from pyspark.sql.functions import year, month, to_date, col, round as spark_round

import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

spark=SparkSession.builder \
  .appName('Day4_BigData_Sales') \
  .config('Spark.sql.adaptive.enabled', 'true') \
  .getOrCreate()

print(f"Spark version : {spark.version}")
print(f"SparkSession : ACTIVE")
print(f"Application : {spark.sparkContext.appName}")

Spark version : 4.0.2
SparkSession : ACTIVE
Application : Day4_BigData_Sales


# BIG DATA AND PySPARK

In [ ]:
df_bronze =spark.read \
   .option('header','true') \
   .option('inferSchema','true') \
   .csv("large_sales_data.csv")

print("=== BRONZE LAYER -Raw Data ===")
print(f'Rows : {df_bronze.count()}')

print(f'Columns : {len(df_bronze.columns)}')
print(f"Names : {df_bronze.columns}")
print()
df_bronze.printSchema()

=== BRONZE LAYER -Raw Data ===
Rows : 5000
Columns : 13
Names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
print("First 5 rows")
df_bronze.show(5,truncate=False)

print('\nBasic statistics for numerical columns')
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Kum

In [ ]:
df_bronze.write \
  .mode('overwrite') \
  .parquet('sales_bronze.parquet')

print('Bronze Parquet saved: sales_bronze.parquet')
import os


Bronze Parquet saved: sales_bronze.parquet


In [ ]:
import pandas as pd
b=pd.read_csv("/content/data.csv")

In [ ]:
b.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA


In [ ]:
X = b[['bedrooms', 'bathrooms', 'sqft_living', 'floors']]
Y = b[['price']]

In [ ]:
b.isnull().any()


,0
date,False
price,False
bedrooms,False
bathrooms,False
sqft_living,False
sqft_lot,False
floors,False
waterfront,False
view,False
condition,False


In [ ]:
b.isnull().sum()

,0
date,0
price,0
bedrooms,0
bathrooms,0
sqft_living,0
sqft_lot,0
floors,0
waterfront,0
view,0
condition,0


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2)

In [ ]:
x_train

,bedrooms,bathrooms,sqft_living,floors
950,3.0,2.25,1990,2.0
4109,2.0,2.00,1170,1.5
2302,3.0,1.75,1520,1.0
1364,4.0,2.25,1940,1.0
495,5.0,3.00,4530,1.5
...,...,...,...,...
1586,4.0,2.75,3310,2.0
1479,7.0,4.50,4140,1.0
3874,3.0,2.50,1840,1.0
2172,3.0,3.50,1500,2.0


In [ ]:
from sklearn.linear_model import LinearRegression
model=LinearRegression()
model.fit(x_train,y_train)

LinearRegression()

In [ ]:
model.predict(x_test)

array([[ 596919.5036551 ],
       [ 365462.40890321],
       [ 454455.84975199],
       [ 391354.48086083],
       [ 701053.86462519],
       [ 875996.44099961],
       [ 404301.31108167],
       [ 630019.974611  ],
       [ 398135.18361072],
       [ 757226.58062875],
       [ 273879.00427923],
       [ 238080.21889901],
       [ 581637.21431434],
       [ 521970.63989948],
       [1021926.31946527],
       [ 185026.93264991],
       [ 437537.37424805],
       [ 359044.56145773],
       [ 679327.29018687],
       [ 540630.65988749],
       [ 509529.23039917],
       [ 486198.29217133],
       [ 808791.63085238],
       [ 615609.75753269],
       [ 650393.90523434],
       [ 422281.52255937],
       [ 467680.83759944],
       [1126823.5981394 ],
       [ 524532.96982593],
       [ 390649.82318251],
       [ 623038.44641925],
       [ 305258.59139416],
       [ 419835.32039697],
       [ 879599.84625671],
       [ 567593.93530815],
       [1563930.02107597],
       [ 434301.81672954],
 

In [ ]:
y_pred=model.predict(x_test)

In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [ ]:
mean_absolute_error(y_test,y_pred)

206655.53056815045

In [ ]:
mean_squared_error(y_test,y_pred)

827962253230.9869

In [ ]:
r2_score(y_test,y_pred)

0.0689201600521615

In [ ]:
import joblib

In [ ]:
joblib.dump(model,"multiple.pkl")

['multiple.pkl']

In [ ]:
df_bronze.write \
  .mode('overwrite') \
  .parquet('sales_bronze.parquet')

print("Bronze Parquet saved : sales_bronze.parquet")

import os

def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path)/1024
  total=0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      fp=os.path.join(dirpath, f)
      total+=os.path.getsize(fp)/1024
  return total

Bronze Parquet saved : sales_bronze.parquet


In [ ]:

df_silver=df_bronze \
  .dropDuplicates() \
  .dropna(subset=['order_id','product','revenue'])

df_silver=df_silver.withColumn(
      'order_date',
      to_date(col('order_date'),'yyyy-MM-dd')
)

df_silver=df_silver \
    .withColumn('order_year',year(col('order_date'))) \
    .withColumn('order_month',month(col('order_date')))



df_silver=df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue')>40000,'High')
    .when(col('revenue')>10000,'Medium')
    .otherwise('Low')
)

print(f"Silver layer rows : {df_silver.count()}")
print('New columns added: order_year, order_month, revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue_category').show(10)

Silver layer rows : 5000
New columns added: order_year, order_month, revenue_category
+----------+-------+----------+-----------+----------------+
|   product|revenue|order_year|order_month|revenue_category|
+----------+-------+----------+-----------+----------------+
|  Keyboard|  13200|      2023|          2|          Medium|
|    Webcam|  17500|      2023|          1|          Medium|
|   Speaker|  58500|      2023|          4|            High|
|  Keyboard|   9600|      2023|         12|             Low|
|    Laptop| 180000|      2023|          8|            High|
|Headphones|  38500|      2023|          5|          Medium|
|    Webcam|  35000|      2023|         11|          Medium|
|    Laptop| 360000|      2023|          1|            High|
|    Tablet| 320000|      2023|          6|            High|
|    Laptop| 225000|      2023|          6|            High|
+----------+-------+----------+-----------+----------------+
only showing top 10 rows


In [ ]:
import os
df_silver.write \
  .mode('overwrite') \
  .parquet('sales_silver.parquet')

print("Silver Parquet saved : sales_silver.parquet")
print(f"Silver size : {get_dir_size("sales_silver.parquet")} KB")

df_verify=spark.read.parquet('sales_silver.parquet')
print(f"Read-back rows : {df_verify.count()} (should match silver count)")
df_verify.printSchema()

Silver Parquet saved : sales_silver.parquet
Silver size : 59.814453125 KB
Read-back rows : 5000 (should match silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)



In [ ]:
top_products = df_silver \
  .groupBy('product') \
  .agg(
      F.sum('revenue').alias('total_revenue'),
      F.count('order_id').alias('num_orders'),
      F.avg('revenue').alias('avg_order_revenue')
  ) \
  .orderBy('total_revenue' , ascending=False) \
  .limit(5)
top_products.show()

+-------+-------------+----------+------------------+
|product|total_revenue|num_orders| avg_order_revenue|
+-------+-------------+----------+------------------+
| Laptop|    182700000|       502|363944.22310756973|
| Tablet|    135104000|       532| 253954.8872180451|
|Monitor|     82126000|       481|170740.12474012474|
|Printer|     44544000|       488| 91278.68852459016|
|Speaker|     16317000|       470| 34717.02127659575|
+-------+-------------+----------+------------------+



In [ ]:
from pyspark.sql import functions as F

revenue_by_region = df_silver \
    .groupBy('region') \
    .agg(
        F.sum('revenue').alias('total_revenue'),
        F.count('order_id').alias('num_orders'),
        F.avg('revenue').alias('avg_order_revenue')
    ) \
    .orderBy('total_revenue', ascending=False)

revenue_by_region.show(truncate = False)

In [ ]:
print(' === Payment Method sucessfully')

In [ ]:
# Gold Table 1: Revenue by region

gold_region = region_revenue
revenue_by_region.write.mode("overwrite").parquet("gold/revenue_by_region")
print("Gold ! saved: gold_region_revenue.parquet")

#Gold Table 2: Product performance summary

gold_products = df_silver \

In [ ]:

revenue_pd = spark.read.parquet("gold_region_revenue.parquet").toPandas()

product_pd = spark.read.parquet("gold_product_summary").toPandas()

monthly_pd = spark.read.parquet("gold_monthly_trend.paruqet").toPandas()


print("Gold tables converted to Pandas:")
print(f"   Region: {region_pd.shape}")
print(f"   Product: {product_pd.shape}")
print(f"   Monthly: {monthly_pd.shape}")

region_pd = region_pd.sort_values('total_revenue',ascendin=False)
product_pd = product_pd.sort_values('total_revenue',ascending=False)
monthly_pd = monthly_pd.sort_values('order_moth')

In [ ]:
fig, axes = plt.subplots(2,2, figsize =(18,13))
fig.suptitle(
    'Big Data Sales Dashboard - PySpark'
)

# panel 1: revenue by region

ax1 = axes[0][0]
bars1 = ax1.bar(region_pd['region'],region_pd['total_revenue'], color = colors4[len(region_pd)],wedgecolor='white')
for bar in bar1:
  ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01, f'(bar.gey_height()/le6:.1f)"

ax2= axe[0][1]
top5_prod = product_pd.head(S).sort_values('total_revenue',ascending=False)
bars2 = ax2.barh(top5_prod['product'],top5_prod['total_revenue'],color='#4FC3F7', edgecolor= 'white')

for bar in bars2:
      ax2.text(bar.get_width()*1.01, bar.get_y()+ bar.get_height()/2,
               f'$(bar.get_width()/1e6: .2f)M', va='center',fontsize=0, fontweight='bold',color='#1E2761')

ax2.set_title("Top 5 branches by Revenue", fontsize='0',fontweight='bold')
ax2.set_xlabel('Total Revenue($)')
ax1.set_ylabel('Total Revenue($)')



In [ ]:
# Activity 2: CSV to Parquet Corporation

print("=== Activity 2: File Format Comparison ===")
print()

csv_kb = get_dir_size("large_sales_data.csv")
bronze_db =get_dir_size('sales_bronze.parquet')
silver_db = get_dir_size('sales_silver.parquet')
gold_db = get_dir_size('gold_region_revenue.parquet') + get_dir_size('gold_product_summary.parquet') + get_dir_size('gold_monthly_trend.parquet'))
print(f"{'File':<38}, {'Size(KB)}' :>10},{'vs CSV' :>10}")
